In [ ]:
import numpy as np

class Detector:
    def __init__(self, R=2.0, N=40, r=0.1, a=0.5):
        self.R = float(R)
        self.N = int(N)
        self.r = float(r)
        self.a = float(a)

        self.pmt_coordinates = self._generate_uniform_pmts()

        self.pmt_normals = self.pmt_coordinates / self.R

    def _generate_uniform_pmts(self):
        i = np.arange(self.N, dtype=float)
        y = 1.0 - 2.0 * (i + 0.5) / self.N
        rho = np.sqrt(1.0 - y**2)
        golden_angle = np.pi * (3.0 - np.sqrt(5.0))
        theta = golden_angle * i

        x = rho * np.cos(theta)
        z = rho * np.sin(theta)

        return self.R * np.column_stack((x, y, z))

    def find_pmt_for_point(self, point, eps=1e-6):
        point = np.asarray(point, dtype=float)

        if abs(np.linalg.norm(point) - self.R) > eps:
            return None

        projections = self.pmt_normals @ point

        tangent_vectors = (
            point[None, :]
            - projections[:, None] * self.pmt_normals
        )

        distances = np.linalg.norm(tangent_vectors, axis=1)

        covered = (projections > 0) & (distances <= self.r)

        indices = np.flatnonzero(covered)

        if len(indices) == 0:
            return None

        nearest = indices[np.argmin(distances[indices])]

        return int(nearest)

    def register_photon(self, point, rng=None):
        pmt_index = self.find_pmt_for_point(point)

        if pmt_index is None:
            return None

        if rng is None:
            rng = np.random.default_rng()

        detected = bool(rng.random() < self.a)

        return pmt_index, detected

def test_detector():
    detector = Detector()

    assert detector.pmt_coordinates.shape == (40, 3)

    radii = np.linalg.norm(detector.pmt_coordinates, axis=1)
    assert np.allclose(radii, detector.R)

    point = detector.pmt_coordinates[0]
    assert detector.find_pmt_for_point(point) == 0
    assert detector.find_pmt_for_point([0, 0, 0]) is None
    assert detector.find_pmt_for_point([2, 0, 0]) is None

    certain_detector = Detector(a=1.0)
    rng = np.random.default_rng(42)

    result = certain_detector.register_photon(
        certain_detector.pmt_coordinates[0],
        rng=rng
    )

    assert result == (0, True)

    print("Все тесты пройдены успешно")

if __name__ == "__main__":
    detector = Detector()

    print("Параметры детектора:")
    print(f"R = {detector.R} м")
    print(f"N = {detector.N}")
    print(f"r = {detector.r} м")
    print(f"a = {detector.a}")

    print("\nПервые 3 ФЭУ:")
    print(detector.pmt_coordinates[:3])

    test_detector()


Параметры детектора:
R = 2.0 м
N = 40
r = 0.1 м
a = 0.5

Первые 3 ФЭУ:
[[ 0.44440972  1.95        0.        ]
 [-0.56035183  1.85        0.51332818]
 [ 0.08464959  1.75       -0.96453846]]
Все тесты пройдены успешно
